# UAT Evidence Pack Runner (GCT-090)

Executes 17 UAT checks and writes per-test evidence artifacts under `/outputs/uat/<test_id>/`.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import re
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent
OUTPUTS = ROOT / 'outputs'
UAT_OUT = OUTPUTS / 'uat'
UAT_OUT.mkdir(parents=True, exist_ok=True)

receipt = (OUTPUTS / 'dataset_receipt.md').read_text(encoding='utf-8')
seed_match = re.search(r'- seed:\s*(\d+)', receipt)
run_date_match = re.search(r'- run_date:\s*([0-9]{4}-[0-9]{2}-[0-9]{2})', receipt)
seed = seed_match.group(1) if seed_match else 'UNKNOWN'
run_date = run_date_match.group(1) if run_date_match else 'UNKNOWN'
print(f'seed={seed} run_date={run_date}')


seed=42 run_date=2026-05-10


In [2]:
commands = []
commands.append([sys.executable, str(ROOT / 'src' / 'verify_dataset.py'), '--run_date', run_date])
commands.append([sys.executable, str(ROOT / 'src' / 'verify_outputs.py')])
command_logs = []
for cmd in commands:
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    command_logs.append((cmd, proc.stdout.strip()))
    cmd_label = Path(cmd[1]).name + (' ' + ' '.join(cmd[2:]) if len(cmd) > 2 else '')
    print('OK:', cmd_label)
print('verification commands complete')


OK: verify_dataset.py --run_date 2026-05-10
OK: verify_outputs.py
verification commands complete


In [3]:
rejects = pd.read_csv(OUTPUTS / 'rejects.csv', dtype=str, keep_default_na=False)
triage = pd.read_csv(OUTPUTS / 'triage_summary.csv', dtype=str, keep_default_na=False)
triage['reject_count'] = pd.to_numeric(triage['reject_count'], errors='coerce').fillna(0).astype(int)
triage['affected_claims'] = pd.to_numeric(triage['affected_claims'], errors='coerce').fillna(0).astype(int)
print(f'reject rows={len(rejects)} triage rows={len(triage)}')


reject rows=219 triage rows=9


In [4]:
TESTS = [
    {'test_id':'UAT-001','rule_ids':['R001_NULL_MEMBER_ID'],'reject_codes':['NULL_MEMBER_ID'],'category':'REQUIRED_FIELD','severity':'BLOCKER'},
    {'test_id':'UAT-002','rule_ids':['R002_NULL_CLAIM_ID'],'reject_codes':['NULL_CLAIM_ID'],'category':'REQUIRED_FIELD','severity':'BLOCKER'},
    {'test_id':'UAT-003','rule_ids':['R003_NULL_PROVIDER_NPI'],'reject_codes':['NULL_PROVIDER_NPI'],'category':'REQUIRED_FIELD','severity':'BLOCKER'},
    {'test_id':'UAT-004','rule_ids':['R004_NULL_SERVICE_FROM'],'reject_codes':['NULL_SERVICE_FROM'],'category':'REQUIRED_FIELD','severity':'BLOCKER'},
    {'test_id':'UAT-005','rule_ids':['R005_NPI_BAD_LENGTH'],'reject_codes':['NPI_BAD_LENGTH'],'category':'PROVIDER','severity':'BLOCKER'},
    {'test_id':'UAT-006','rule_ids':['R006_NPI_NOT_NUMERIC'],'reject_codes':['NPI_NOT_NUMERIC'],'category':'PROVIDER','severity':'BLOCKER'},
    {'test_id':'UAT-007','rule_ids':['R007_NPI_NOT_FOUND'],'reject_codes':['NPI_NOT_FOUND'],'category':'PROVIDER','severity':'BLOCKER'},
    {'test_id':'UAT-008','rule_ids':['R008_SERVICE_TO_BEFORE_FROM'],'reject_codes':['SERVICE_TO_BEFORE_FROM'],'category':'DATES','severity':'BLOCKER'},
    {'test_id':'UAT-009','rule_ids':['R009_FUTURE_SERVICE_DATE'],'reject_codes':['FUTURE_SERVICE_DATE'],'category':'DATES','severity':'BLOCKER'},
    {'test_id':'UAT-010','rule_ids':['R010_SERVICE_OUTSIDE_COVERAGE'],'reject_codes':['SERVICE_OUTSIDE_COVERAGE'],'category':'MEMBER_ELIGIBILITY','severity':'HIGH'},
    {'test_id':'UAT-011','rule_ids':['R011_DUP_CLAIM_KEY'],'reject_codes':['DUP_CLAIM_KEY'],'category':'DUPLICATE','severity':'HIGH'},
    {'test_id':'UAT-012','rule_ids':['R012_PAID_GT_ALLOWED'],'reject_codes':['PAID_GT_ALLOWED'],'category':'FINANCIAL','severity':'MONITOR'},
    {'test_id':'UAT-013','rule_ids':['R013_ALLOWED_GT_CHARGE'],'reject_codes':['ALLOWED_GT_CHARGE'],'category':'FINANCIAL','severity':'MONITOR'},
    {'test_id':'UAT-014','rule_ids':['R014_NULL_PROC','R015_NULL_DX'],'reject_codes':['NULL_PROC','NULL_DX'],'category':'CODE_SET','severity':'MONITOR'},
    {'test_id':'UAT-015','rule_ids':['R901_DUP_RATE_GT_1PCT'],'reject_codes':['DUP_RATE_GT_1PCT'],'category':'DUPLICATE','severity':'HIGH','batch_convention':True},
    {'test_id':'UAT-016','rule_ids':['R902_ELIG_MISMATCH_GT_2PCT'],'reject_codes':['ELIG_MISMATCH_GT_2PCT'],'category':'MEMBER_ELIGIBILITY','severity':'HIGH','batch_convention':True},
    {'test_id':'UAT-017','rule_ids':['R903_VOLUME_SHIFT_GT_15PCT'],'reject_codes':['VOLUME_SHIFT_GT_15PCT'],'category':'VOLUME_ANOMALY','severity':'MONITOR','batch_convention':True},
]

results = []
ts = datetime.now(timezone.utc).isoformat()
for t in TESTS:
    test_id = t['test_id']
    out_dir = UAT_OUT / test_id
    out_dir.mkdir(parents=True, exist_ok=True)

    rej = rejects[rejects['reject_code'].isin(t['reject_codes'])].copy()
    tri = triage[(triage['reject_category'] == t['category']) & (triage['severity'] == t['severity'])].copy()

    assert len(rej) > 0, f'{test_id} expected rejects rows for {t["reject_codes"]}'
    assert int(tri['reject_count'].sum()) > 0, f'{test_id} expected triage impact for {t["category"]}/{t["severity"]}'

    if 'NULL_CLAIM_ID' in t['reject_codes']:
        assert (rej['claim_id'].str.strip() == '').all(), f'{test_id} expected blank claim_id for NULL_CLAIM_ID'

    if t.get('batch_convention'):
        assert rej['claim_id'].str.startswith('BATCH::').all(), f'{test_id} batch convention claim_id failed'
        assert (rej['line_id'].fillna('').str.strip() == '').all(), f'{test_id} batch convention line_id failed'
        assert (rej['service_from'].fillna('').str.strip() == '').all(), f'{test_id} batch convention service_from failed'

    rej.to_csv(out_dir / 'rejects.csv', index=False)
    tri.to_csv(out_dir / 'triage_summary.csv', index=False)

    log_lines = [
        f'timestamp_utc: {ts}',
        f'test_id: {test_id}',
        f'rule_ids: {",".join(t["rule_ids"])}',
        f'reject_codes: {",".join(t["reject_codes"])}',
        f'seed: {seed}',
        f'run_date: {run_date}',
        'commands:'
    ]
    for cmd, out in command_logs:
        log_lines.append('  - ' + ' '.join(cmd))
        first_line = out.splitlines()[0] if out else ''
        log_lines.append('    output_head: ' + first_line)
    log_lines.extend([
        f'reject_rows: {len(rej)}',
        f'triage_rows: {len(tri)}',
        f'triage_reject_count_sum: {int(tri["reject_count"].sum())}',
    ])
    (out_dir / 'run_log.txt').write_text('\n'.join(log_lines) + '\n', encoding='utf-8')

    results.append({
        'test_id': test_id,
        'rule_ids': ','.join(t['rule_ids']),
        'reject_rows': len(rej),
        'triage_rows': len(tri),
        'triage_reject_count_sum': int(tri['reject_count'].sum()),
        'evidence_path': out_dir.relative_to(ROOT).as_posix(),
    })

results_df = pd.DataFrame(results).sort_values('test_id').reset_index(drop=True)
results_df.to_csv(UAT_OUT / 'uat_results_summary.csv', index=False)
print(results_df)
summary_path = UAT_OUT / 'uat_results_summary.csv'
print(f'Wrote summary: {summary_path.relative_to(ROOT).as_posix()}')


    test_id                       rule_ids  reject_rows  triage_rows  \
0   UAT-001            R001_NULL_MEMBER_ID           10            1   
1   UAT-002             R002_NULL_CLAIM_ID            5            1   
2   UAT-003         R003_NULL_PROVIDER_NPI           10            1   
3   UAT-004         R004_NULL_SERVICE_FROM            5            1   
4   UAT-005            R005_NPI_BAD_LENGTH           10            1   
5   UAT-006           R006_NPI_NOT_NUMERIC           10            1   
6   UAT-007             R007_NPI_NOT_FOUND           10            1   
7   UAT-008    R008_SERVICE_TO_BEFORE_FROM           10            2   
8   UAT-009       R009_FUTURE_SERVICE_DATE           10            2   
9   UAT-010  R010_SERVICE_OUTSIDE_COVERAGE            6            1   
10  UAT-011             R011_DUP_CLAIM_KEY           10            1   
11  UAT-012           R012_PAID_GT_ALLOWED           40            1   
12  UAT-013         R013_ALLOWED_GT_CHARGE           40         